In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from plotly import express as px
from plotly import graph_objects as go

In [ ]:
flat_df=pd.read_csv("gandhinagar_property_apartments.csv")
house_df=pd.read_csv("gandhinagar_property_houses.csv")
plot_df=pd.read_csv("gandhinagar_property_plots.csv")

In [3]:
print(flat_df.shape)
print(house_df.shape)
print(plot_df.shape)

(1319, 23)
(229, 23)
(132, 23)


In [5]:
print(flat_df.duplicated().sum())
print(house_df.duplicated().sum())
print(plot_df.duplicated().sum())

1
0
0


In [8]:
flat_df[flat_df.duplicated(keep=False)]

,Name,Location,Price (INR in Lakhs),Price_per_sqft,Area_sqft,Description,Property_URL,bedrooms,bathrooms,balconies,...,Mapped_Area,facing,property_age_bucket,is_ready_to_move,is_built_up_area,is_carpet_area,is_super_built_up_area,property_type,luxury_score,location_advantage_score
53,"3 BHK Flatin Sargasan, Gandhinagar","Sargasan, Gandhinagar",75.0,9475.0,791.0,The flat is facing the east direction. Constru...,https://www.99acres.com/3-bhk-bedroom-apartmen...,3.0,3.0,1.0,...,Sargasan,East,New,1,False,True,False,apartment,0.0,0.0
63,"3 BHK Flatin Sargasan, Gandhinagar","Sargasan, Gandhinagar",75.0,9475.0,791.0,The flat is facing the east direction. Constru...,https://www.99acres.com/3-bhk-bedroom-apartmen...,3.0,3.0,1.0,...,Sargasan,East,New,1,False,True,False,apartment,0.0,0.0


In [10]:
flat_df.drop_duplicates(inplace=True)

In [65]:
def numeric_eda(df, col):

    print("="*60)
    print(f"EDA OF {col.upper()}")
    print("="*60)

    # Missing values
    print("\nMissing Values :", df[col].isnull().sum())

    # Basic statistics
    print("\nSummary Statistics")
    print(df[col].describe())

    # Distribution
    plt.figure(figsize=(10,5))

    sns.histplot(
        df[col],
        kde=True
    )

    plt.title(f"{col} Distribution")
    plt.xlabel(col)
    plt.ylabel("Frequency")

    plt.show()

    # Boxplot
    plt.figure(figsize=(10,3))

    sns.boxplot(x=df[col])

    plt.title(f"Boxplot of {col}")

    plt.show()

    # Skewness & Kurtosis
    skewness = df[col].skew()
    kurtosis = df[col].kurtosis()

    print("\nSkewness :", round(skewness, 3))
    print("Kurtosis :", round(kurtosis, 3))

    # IQR Outliers
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)

    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    print("\nLower Bound :", round(lower_bound, 2))
    print("Upper Bound :", round(upper_bound, 2))

    outliers = df[
        (df[col] < lower_bound) |
        (df[col] > upper_bound)
    ]

    print("\nOutlier Count :", len(outliers))
    print("Outlier Percentage :",
          round(len(outliers)/len(df)*100, 2), "%")

    if len(outliers) > 0:
        print("\nOutlier Statistics")
        print(outliers[col].describe())

    # Original vs Log Distribution
    plt.figure(figsize=(12,6))

    plt.subplot(1,2,1)
    sns.histplot(
        df[col],
        kde=True,
        color='green'
    )
    plt.title("Original Distribution")

    plt.subplot(1,2,2)
    sns.histplot(
        np.log1p(df[col]),
        kde=True,
        color='blue'
    )
    plt.title("Log-Transformed Distribution")

    plt.tight_layout()
    plt.show()

    return outliers

In [66]:
def discrete_numeric_eda(df, col):

    print("="*60)
    print(f"EDA OF {col.upper()}")
    print("="*60)

    # Missing Values
    print("\nMissing Values :", df[col].isnull().sum())

    print(
        "Missing Percentage :",
        round(df[col].isnull().mean() * 100, 2),
        "%"
    )

    # Summary Statistics
    print("\nSummary Statistics")
    print(df[col].describe())

    # Value Counts
    print("\nValue Counts")
    print(df[col].value_counts().sort_index())

    # Count Plot
    plt.figure(figsize=(10,5))

    sns.countplot(
        x=df[col]
    )

    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Count")

    plt.show()

    # Pie Chart
    counts = df[col].value_counts()

    plt.figure(figsize=(8,8))

    plt.pie(
        counts.values,
        labels=counts.index,
        autopct='%1.1f%%'
    )

    plt.title(f"{col} Distribution")

    plt.show()

## Location

In [23]:
def location_eda(df):

    print("="*60)
    print("LOCATION EDA")
    print("="*60)

    # Unique locations
    print("\nUnique Locations :", df['Location'].nunique())

    # Missing values
    print("\nMissing Values :", df['Location'].isnull().sum())

    # Top 15 locations
    top_locations = df['Location'].value_counts().head(15)

    plt.figure(figsize=(12,6))

    sns.barplot(
        x=top_locations.index,
        y=top_locations.values
    )

    plt.xticks(rotation=90)
    plt.title("Top 15 Locations")
    plt.xlabel("Location")
    plt.ylabel("Count")
    plt.show()

    # Location percentage
    location_percent = (
        df['Location'].value_counts(normalize=True) * 100
    )

    print("\nTop 10 Locations (%)")
    print(location_percent.head(10))

    # Rare locations (<5 properties)
    location_counts = df['Location'].value_counts()
    # rare_locations = location_counts[location_counts < 5]
    rare_locations = location_counts[location_counts < 3]
    print("\nRare Locations Count :", len(rare_locations))

    # Cumulative percentage
    cumulative = (
        df['Location']
        .value_counts(normalize=True)
        .cumsum() * 100
    )

    print("\nCumulative Percentage")
    print(cumulative.head(10))

    return {
        "top_locations": top_locations,
        "location_percent": location_percent,
        "rare_locations": rare_locations,
        "cumulative": cumulative
    }

In [25]:
# print("Apartment Dataset")
# location_eda(flat_df)

# print("House Dataset")
# location_eda(house_df)

# print("Plot Dataset")
# location_eda(plot_df)

### Observations

#### Apartment Dataset

* The apartment dataset contains **69 unique locations**.
* There are **no missing values** in the `Location` column.
* **Sargasan** is the most frequent location with **412 properties**, accounting for approximately **30%** of all apartment listings.
* The top **9 locations** collectively account for nearly **85%** of the apartment dataset, indicating a high concentration of listings in a few key areas.
* There are **50 rare locations** with fewer than **5 listings** each.

#### House/Villa Dataset

* The house/villa dataset contains **64 unique locations**.
* There are **no missing values** in the `Location` column.
* **Kudasan** is the most frequent location with **28 properties**, representing approximately **12%** of the dataset.
* The top **10 locations** together contribute around **60%** of all house/villa listings, suggesting a more balanced geographical distribution compared to apartments.
* There are **53 rare locations** with fewer than **5 listings** each.

#### Plot Dataset

* The plot dataset contains **54 unique locations**.
* There are **no missing values** in the `Location` column.
* **Palodia** is the most frequent location with **21 properties**, accounting for approximately **15%** of all plot listings.
* The top **9 locations** cover roughly **50%** of the dataset, indicating that plot listings are more geographically dispersed than apartments and houses.
* There are **39 rare locations** with fewer than **3 listings** each.

## Price (INR in Lakhs)

In [38]:
# numeric_eda(flat_df, col='Price (INR in Lakhs)')
# numeric_eda(house_df, col='Price (INR in Lakhs)')
# numeric_eda(plot_df, col='Price (INR in Lakhs)')

### Observations

#### Apartment Dataset

* There are **no missing values** in the `Price (INR in Lakhs)` column.
* The **mean price is ₹95 Lakhs**, while the **median price is ₹75 Lakhs**.
* The mean being noticeably higher than the median indicates the presence of high-priced apartments that pull the average upward.
* The standard deviation is relatively high, suggesting considerable variability in apartment prices.
* The price distribution is **highly right-skewed and leptokurtic**, indicating a long right tail and the presence of several premium and luxury apartments.
* Using the IQR method, **81 observations** were identified as outliers.
* Log transformation substantially reduces the skewness and makes the distribution more symmetric and closer to normal.

#### House/Villa Dataset

* There are **no missing values** in the `Price (INR in Lakhs)` column.
* The **mean price is ₹242 Lakhs**, while the **median price is ₹200 Lakhs**.
* The higher mean compared to the median suggests the presence of expensive villas and luxury houses.
* The standard deviation is very high, indicating substantial variation in property prices.
* The distribution is **moderately right-skewed and leptokurtic**, with a concentration of properties in the lower and middle price ranges and a smaller number of high-value properties extending the upper tail.
* Using the IQR method, **13 observations** were identified as outliers.
* Log transformation reduces skewness and produces a distribution that is more suitable for modeling.

#### Plot Dataset

* There are **no missing values** in the `Price (INR in Lakhs)` column.
* The **mean price is ₹469 Lakhs**, while the **median price is ₹289 Lakhs**.
* The large gap between the mean and median indicates the presence of several extremely high-value land parcels.
* The standard deviation is extremely high and exceeds the mean, highlighting the highly dispersed nature of plot prices.
* The distribution is **highly right-skewed and leptokurtic**, with a strong concentration of observations at lower price levels and a few very expensive plots creating a long right tail.
* Using the IQR method, **12 observations** were identified as outliers.
* Log transformation significantly reduces skewness and improves the overall distribution shape.

## Price_per_sqft

In [45]:
# numeric_eda(flat_df,col='Price_per_sqft')
# numeric_eda(house_df,col='Price_per_sqft')
# numeric_eda(plot_df,col='Price_per_sqft')

### Observations

#### Apartment Dataset

* There are **no missing values** in the `Price_per_sqft` column.
* The **mean Price_per_sqft is approximately ₹8,447**, while the **median is approximately ₹5,000**.
* The standard deviation is very high and exceeds the median, indicating substantial variability in apartment prices per square foot.
* The boxplot reveals a large number of extreme values, suggesting the presence of premium and luxury apartment listings.
* Using the IQR method, **81 observations** were identified as outliers.
* The distribution exhibits **very high positive skewness and kurtosis**, indicating a highly right-skewed and leptokurtic distribution with a long upper tail.
* Applying a **log transformation (`log1p`)** significantly reduces skewness and produces a distribution that is more suitable for statistical analysis and predictive modeling.

#### House/Villa Dataset

* There are **4 missing values** in the `Price_per_sqft` column.
* The **mean Price_per_sqft is approximately ₹20,000**, while the **median is approximately ₹12,000**.
* The standard deviation is extremely high, reflecting considerable variation in prices across houses and villas.
* Several high-value luxury properties contribute to the wide spread of the distribution.
* Using the IQR method, **20 observations** were identified as outliers.
* The distribution is **highly right-skewed and leptokurtic**, indicating the presence of extreme high-value properties.
* Log transformation substantially reduces skewness and improves the overall shape of the distribution.

#### Plot Dataset

* There are **2 missing values** in the `Price_per_sqft` column.
* The **mean Price_per_sqft is approximately ₹7,200**, while the **median is approximately ₹4,800**.
* The standard deviation is extremely high relative to the mean, indicating significant variability in land prices across locations.
* The boxplot shows several extreme observations.
* Using the IQR method, **12 observations** were identified as outliers.
* The distribution is **highly right-skewed and leptokurtic**, with a small number of very high-priced plots creating a long right tail.
* Applying a **log transformation (`log1p`)** helps normalize the distribution and reduces the impact of extreme values.

## Area_sqft

In [49]:
# numeric_eda(flat_df,col='Area_sqft')
# numeric_eda(house_df,col='Area_sqft')
# numeric_eda(plot_df,col='Area_sqft')

### Observations

#### Apartment Dataset

* There are **no missing values** in the `Area_sqft` column.
* The **mean area is approximately 1,822 sqft**, while the **median area is approximately 1,395 sqft**.
* The standard deviation is extremely high and is nearly **three times the mean**, indicating substantial variability in apartment sizes.
* The distribution is **highly right-skewed and leptokurtic**, with a long upper tail caused by a small number of very large residential units.
* The boxplot reveals the presence of several extreme observations.
* Using the IQR method, **63 observations** were identified as outliers.
* Applying a **log transformation** significantly reduces skewness and improves the overall shape of the distribution.

#### House/Villa Dataset

* There are **no missing values** in the `Area_sqft` column.
* The **mean area is approximately 3,090 sqft**, while the **median area is approximately 1,620 sqft**.
* The standard deviation is exceptionally high and is nearly **five times the mean**, indicating a very wide variation in property sizes.
* The distribution is **highly right-skewed and leptokurtic**, reflecting the presence of several large villas and luxury houses.
* The boxplot highlights a number of extreme observations that substantially increase the spread of the data.
* Using the IQR method, **22 observations** were identified as outliers.
* Log transformation effectively reduces skewness and produces a more balanced distribution.

#### Plot Dataset

* There are **no missing values** in the `Area_sqft` column.
* The **mean area is approximately 70,000 sqft**, while the **median area is approximately 5,000 sqft**.
* The large gap between the mean and median indicates the presence of extremely large land parcels.
* The standard deviation is extraordinarily high, reflecting the highly dispersed nature of plot sizes.
* The distribution is **highly right-skewed and leptokurtic**, with a few very large plots creating an extended upper tail.
* The boxplot clearly shows several extreme observations.
* Using the IQR method, **18 observations** were identified as outliers.
* Log transformation significantly reduces skewness and improves the overall distribution shape.


## Luxury_Score  and Location_Advantage_Score

In [112]:
# numeric_eda(flat_df, col='luxury_score')
# numeric_eda(flat_df, col='location_advantage_score')

### Observations

As Description col is not much talking about the luxury and location advantage of the property due to that luxury_score and location_advantage_score are not much useful for our model and we can drop these feature from our dataset.

## Bedrooms

In [ ]:
# discrete_numeric_eda(flat_df, col='bedrooms')
# discrete_numeric_eda(house_df, col='bedrooms')

,Name,Price (INR in Lakhs),Price_per_sqft,Area_sqft,bedrooms
66,"Residential propertyin Mansa, Gandhinagar",520.0,955.0,54455.076,0.0
73,"4 Bedroom Farm housein Dahegam, Gandhinagar",2405.0,955.0,251831.000,4.0
88,"Residential land / Plotin Vavol, Gandhinagar",55.0,5556.0,990.000,0.0
92,"2 Bedroom Farm housein Mahudi, Gandhinagar",50.0,1388.0,3595.176,2.0
97,"Residential land / Plotin Grambharti, Gandhinagar",119.0,912.0,13050.000,0.0


In [64]:
plot_df[~plot_df['bedrooms'].isnull()][['Name','Price (INR in Lakhs)','Price_per_sqft','Area_sqft','bedrooms']]
plot_df[plot_df['Name'].str.contains('Farm')]

,Name,Location,Price (INR in Lakhs),Price_per_sqft,Area_sqft,Description,Property_URL,bedrooms,bathrooms,balconies,...,Mapped_Area,facing,property_age_bucket,is_ready_to_move,is_built_up_area,is_carpet_area,is_super_built_up_area,property_type,luxury_score,location_advantage_score
73,"4 Bedroom Farm housein Dahegam, Gandhinagar","Dahegam, Gandhinagar",2405.0,955.0,251831.000,"This 4 bhk farmhouse in dahegam, gandhinagar o...",https://www.99acres.com/4-bhk-bedroom-farm-hou...,4.0,1.0,2.0,...,Dahegam,NaN,1-5 years,1,False,False,False,farmhouse,0.0,2.0
92,"2 Bedroom Farm housein Mahudi, Gandhinagar","Mahudi, Gandhinagar",50.0,1388.0,3595.176,We are proud owners of farmhouse available for...,https://www.99acres.com/2-bhk-bedroom-farm-hou...,2.0,2.0,2.0,...,Mansa,NaN,New,1,False,False,False,farmhouse,0.0,0.0


### Observations

#### Apartment Dataset

* There are **2 missing values** in the `bedrooms` column.
* The standard deviation is relatively low, indicating that the number of bedrooms is fairly consistent across apartment listings.
* 2-bedroom properties (42%) and 3-bedroom properties (40%) are the most common configurations in the apartment dataset, followed by 1-bedroom and 4-bedroom properties.

#### House/Villa Dataset

* There are **3 missing values** in the `bedrooms` column.
* The standard deviation is relatively low, indicating limited variation in the number of bedrooms among houses and villas.
* 2-bedroom properties (38%) and 3-bedroom properties (26.5%) are the most common configurations in the apartment dataset, followed by 2-bedroom and 5-bedroom properties.

#### Plot Dataset

* The `bedrooms` feature is not relevant for the plot dataset.
* Almost all observations correspond to residential plots where bedroom information is not applicable.
* Only **two farmhouse records** contain valid bedroom counts, while the remaining plot listings have missing bedroom information.
* Due to the lack of meaningful bedroom data and the very small number of farmhouse observations, the `bedrooms` feature was excluded from further analysis and modeling for plot properties.

## Bathrooms

In [ ]:
# discrete_numeric_eda(flat_df, col='bathrooms')
# discrete_numeric_eda(house_df, col='bathrooms')

In [78]:
plot_df[~plot_df['bathrooms'].isnull()][['Name','Price (INR in Lakhs)','Price_per_sqft','Area_sqft','bathrooms']]

,Name,Price (INR in Lakhs),Price_per_sqft,Area_sqft,bathrooms
66,"Residential propertyin Mansa, Gandhinagar",520.0,955.0,54455.076,0.0
73,"4 Bedroom Farm housein Dahegam, Gandhinagar",2405.0,955.0,251831.000,1.0
88,"Residential land / Plotin Vavol, Gandhinagar",55.0,5556.0,990.000,1.0
92,"2 Bedroom Farm housein Mahudi, Gandhinagar",50.0,1388.0,3595.176,2.0
97,"Residential land / Plotin Grambharti, Gandhinagar",119.0,912.0,13050.000,1.0


### Observations

#### Apartment Dataset

* There are **2 missing values** in the `bathrooms` column.
* The standard deviation is relatively low, indicating limited variation in the number of bathrooms across apartment listings.
* **2-bathroom properties (44%) and 3-bathroom properties (37%) are the most common configurations in the apartment dataset, followed by 1-bathroom and 4-bathroom properties.**

#### House/Villa Dataset

* There are **3 missing values** in the `bathrooms` column.
* The standard deviation is relatively low, indicating limited variation in the number of bathrooms across house and villa listings.
* **4-bathroom properties (32%) and 3-bathroom properties (26%) are the most common configurations in the house/villa dataset, followed by 2-bathroom and 5-bathroom properties.**

#### Plot Dataset

* The `bathrooms` feature is **not relevant** for the plot dataset.
* Most observations correspond to residential plots where bathroom information is not applicable.
* Only a small number of farmhouse properties contain valid bathroom information.
* Therefore, the `bathrooms` feature need to be  excluded from further analysis and modeling for plot properties.

## Balacony

In [79]:
# discrete_numeric_eda(flat_df,col='balconies')
# discrete_numeric_eda(house_df,col='balconies')

In [77]:
plot_df[~plot_df['balconies'].isnull()][['Name','Price (INR in Lakhs)','Price_per_sqft','Area_sqft','balconies']]

,Name,Price (INR in Lakhs),Price_per_sqft,Area_sqft,balconies
66,"Residential propertyin Mansa, Gandhinagar",520.0,955.0,54455.076,0.0
73,"4 Bedroom Farm housein Dahegam, Gandhinagar",2405.0,955.0,251831.000,2.0
88,"Residential land / Plotin Vavol, Gandhinagar",55.0,5556.0,990.000,0.0
92,"2 Bedroom Farm housein Mahudi, Gandhinagar",50.0,1388.0,3595.176,2.0
97,"Residential land / Plotin Grambharti, Gandhinagar",119.0,912.0,13050.000,0.0


### Observations

#### Apartment Dataset

* There are **2 missing values** in the `balconies` column.
* The standard deviation is relatively low, indicating limited variation in the number of balconies across apartment listings.
* **Properties with 1 balcony are the most common, accounting for approximately 70% of the apartment dataset, followed by properties with 2 balconies.**


#### House/Villa Dataset

* There are **3 missing values** in the `balconies` column.
* The standard deviation is relatively low, indicating limited variation in the number of balconies across house and villa listings.
* **Properties with 2 balconies are the most common, accounting for approximately 50% of the house/villa dataset, followed by properties with 1 balcony.**

#### Plot Dataset

* The `balconies` feature is **not relevant** for the plot dataset.
* Most observations correspond to residential plots where balcony information is not applicable.
* Therefore, the `balconies` feature need to be  excluded from further analysis and modeling for plot properties


## Current_floor

Index 12 is a flat but was mistakenly placed in `house_df` instead of `apartment_df`.  
  → Move index 12 to `apartment_df` and remove it from `house_df`.

In [100]:
# discrete_numeric_eda(flat_df, col='current_floor')
# discrete_numeric_eda(house_df, col='current_floor')

house_df[~house_df['current_floor'].isnull()]['current_floor'].value_counts()
# house_df.loc[12]

current_floor
2.0    6
0.0    4
1.0    3
5.0    1
Name: count, dtype: int64

In [101]:
# discrete_numeric_eda(plot_df,col='current_floor')

plot_df[~plot_df['current_floor'].isnull()][['current_floor']]

,current_floor
14,0.0
76,0.0


### Observations

#### Apartment Dataset

- **Missing values:** 548  
- **Standard deviation:** moderate, same as mean  

- **Floor distribution:**  
  Most properties are located between the ground floor and the 6th floor.  
  The **2nd floor** has the highest number of properties.  
  A few properties are located on higher floors, but their frequency is very low.


#### House / Villa Dataset

These feature is **not relevant** for the House / Villa Dataset.

#### Plot Dataset

These feature is **not relevant** for the Plot Dataset.

## Total_floors

In [109]:
# discrete_numeric_eda(flat_df, col='total_floors')
# discrete_numeric_eda(house_df, col='total_floors')

In [ ]:
# discrete_numeric_eda(plot_df,col='total_floors')

plot_df[~plot_df['total_floors'].isnull()][['Name','total_floors']]

,Name,total_floors
25,"Residential land / Plotin Palodia, Gandhinagar",3.0
26,"Residential land / Plotin Palodia, Gandhinagar",3.0
40,"Residential land / Plotin Unali, Gandhinagar",2.0
44,"Residential land / Plotin Palodia, Gandhinagar",2.0
53,"Residential land / Plotin Koba, Gandhinagar",2.0
55,"Residential land / Plotin Sargasan, Gandhinagar",10.0
77,"Residential land / Plotin Palodia, Gandhinagar",2.0
83,"Residential land / Plotin Palodia, Gandhinagar",2.0
86,"Residential land / Plotin Vavol, Gandhinagar",6.0
87,"Residential land / Plotin Vavol, Gandhinagar",4.0


### Observations

#### Apartment Dataset

- **Missing values:** 603
- **Standard deviation:** low to moderate
- **Mean vs. median:** mean is near the same as median
- **Floor frequency distribution:**  
  The 5th and 13th floors have the highest frequency for `total_floors`, followed by the 7th and 14th floors.

#### House / Villa Dataset

- **Missing values:** 216 (approximately 95% are missing)
- **Standard deviation:** low
- **Relevance:**  
  The `total_floors` feature is **not relevant** for the house/villa dataset because it indicates the total number of floors in a building, which is applicable only to multi‑story apartments – not to standalone houses or villas.

#### Plot Dataset

- **Relevance:**  
  The `total_floors` feature is **not relevant** for the plot dataset as well, since plots are vacant land with no existing structure.

## Furnishing_Status 

In [147]:
def categorical_eda(df, col):

    print("="*60)
    print(f"{col.upper()} EDA")
    print("="*60)

    # Missing values
    print(f"\nMissing Values in {col} :", df[col].isnull().sum())

    # Unique values
    print(f"\nUnique Values in {col} :", df[col].unique())

    # Value counts
    print(f"\nValue Counts of {col} :")
    print(df[col].value_counts())

    # Percentage distribution
    dist_percent = df[col].value_counts(normalize=True) * 100
    print(f"\nPercentage Distribution (%) of {col}:")
    print(dist_percent)

    # Plot
    plt.figure(figsize=(8,5))
    sns.countplot(
        x=df[col],
        order=df[col].value_counts().index
    )

    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.xticks(rotation=45)
    plt.show()


In [ ]:
categorical_eda(flat_df, col='furnishing_status')
categorical_eda(house_df, col='furnishing_status')
categorical_eda(plot_df, col='furnishing_status')

Take idx 73 and 92  to house_df from plot_df

In [124]:
# plot_df[~plot_df['furnishing_status'].isnull()][['Name','Price (INR in Lakhs)','Price_per_sqft','Area_sqft','furnishing_status']]

plot_df[plot_df['Name'].str.contains('Farm')][['Name','Price (INR in Lakhs)','Price_per_sqft','Area_sqft','furnishing_status']]

,Name,Price (INR in Lakhs),Price_per_sqft,Area_sqft,furnishing_status
73,"4 Bedroom Farm housein Dahegam, Gandhinagar",2405.0,955.0,251831.000,furnished
92,"2 Bedroom Farm housein Mahudi, Gandhinagar",50.0,1388.0,3595.176,NaN


### Observations

#### Furnishing Status

There are **3 types** of furnishing status in the dataset:  
- Furnished  
- Semi‑Furnished  
- Unfurnished  

#### Apartment Dataset

- **Missing values:** 687  
- **Distribution:** Unfurnished properties are the most common, covering approximately **65%** of the apartment dataset.

#### House / Villa Dataset

- **Missing values:** 152  
- **Distribution:** Unfurnished properties are also the most common, covering approximately **62%** of the house/villa dataset.

## Mapped_Area

In [136]:
def mapped_area_eda(df):

    col = "Mapped_Area"

    print("="*60)
    print("MAPPED AREA EDA")
    print("="*60)

    # Unique values
    print(f"\nUnique {col} :", df[col].nunique())

    # Missing values
    print(f"\nMissing Values :", df[col].isnull().sum())

    # Top 15 mapped areas
    top_areas = df[col].value_counts().head(15)

    plt.figure(figsize=(12,6))
    sns.barplot(
        x=top_areas.index,
        y=top_areas.values
    )

    plt.xticks(rotation=90)
    plt.title("Top 15 Mapped Areas")
    plt.xlabel("Mapped Area")
    plt.ylabel("Count")
    plt.show()
    
    # value counts
    print(f"\nValue Counts of {col} :")
    print(df[col].value_counts())

    # Percentage distribution
    area_percent = df[col].value_counts(normalize=True) * 100

    print("\nTop 10 Mapped Areas (%)")
    print(area_percent.head(10))

    # Rare areas (<3 properties)
    area_counts = df[col].value_counts()
    rare_areas = area_counts[area_counts < 3]

    print("\nRare Mapped Areas Count :", len(rare_areas))

    # Cumulative percentage
    cumulative = area_percent.cumsum()

    print("\nCumulative Percentage")
    print(cumulative.head(10))


In [193]:
# mapped_area_eda(flat_df)
# mapped_area_eda(house_df)
# mapped_area_eda(plot_df)

### Observations

#### Apartment Dataset

- **Unique mapped areas:** 36
- **Missing values:** 0
- **Most frequent mapped area:** Sargasan – 421 properties, accounting for approximately **30%** of all apartment listings.
- **Top locations coverage:** 9 locations cover **90%** of the apartment dataset.
- **Rare areas:** 16 areas have less than 5 listings each.

#### House / Villa Dataset

- **Unique mapped areas:** 37
- **Missing values:** 0
- **Most frequent mapped area:** Kudasan – 28 properties, representing approximately **12%** of the house/villa dataset.
- **Top locations coverage:** 10 locations cover **70%** of the house/villa dataset.
- **Rare areas:** 24 areas have less than 5 listings each.

#### Plot Dataset

- **Unique mapped areas:** 26
- **Missing values:** 0
- **Most frequent mapped area:** Palodia – 21 properties, accounting for approximately **15%** of all plot listings.
- **Top locations coverage:** 10 locations cover **75%** of the plot dataset.
- **Rare areas:** 10 areas have less than 3 listings each.

## Facing

In [144]:
# categorical_eda(flat_df, col='facing')
# categorical_eda(house_df, col='facing')
# categorical_eda(plot_df, col='facing')

### Observations

**Facing directions present in the dataset:**  
East, Other, North-East, North-West, South, North, West, South-West, South-East

#### Apartment Dataset

- **Missing values:** 860  
- **Most common facing direction:** East – covers approximately **62%** of the apartment dataset.

#### House / Villa Dataset

- **Missing values:** 222  
- **Most common facing directions:** East and North are the most common.  
- **Note:** Data for the facing feature is largely missing in the house/villa dataset.

#### Plot Dataset

- **Missing values:** 120  
- **Most common facing direction:** East – covers approximately **65%** of the plot dataset.

#### Recommendation
 
- Therefore, the `facing` feature can be **dropped** from both the **plot dataset** and the **house/villa dataset** for further analysis and modeling.

## Property_Age_Bucket

In [192]:
# categorical_eda(flat_df, col='property_age_bucket')
# categorical_eda(house_df, col='property_age_bucket')
# categorical_eda(plot_df, col='property_age_bucket')

### Observations

**Categories present:**  
New, 5-10 years, 1-5 years, Unknown, 10-20 years

#### Apartment Dataset

- **Missing values:** 0  
- **Highest frequency:** `New` – covers approximately **47%** of the apartment dataset.

#### House / Villa Dataset

- **Missing values:** 0  
- **Highest frequency:** `New` – covers approximately **40%** of the house/villa dataset.

#### Plot Dataset

- **Missing values:** 0  
- **Highest frequency:** `Unknown` – covers approximately **50%** of the plot dataset.  
- **Issue:** This is problematic for modeling because `Unknown` provides no information about the age of the property, yet it represents half of the plot data.

#### General Trend

As the age of the property increases, the frequency of properties decreases across the **apartment**, **plot**, and **house/villa** datasets.

## Property_Type

In [157]:
# categorical_eda(flat_df, col='property_type')
# categorical_eda(house_df, col='property_type')
# categorical_eda(plot_df, col='property_type')

### Observations

#### Apartment Dataset (flat_df)

- **Missing values:** 0  
- **Frequency:** Apartment has the highest frequency in `flat_df`, covering approximately **95%** of the dataset.

#### House / Villa Dataset

- **Missing values:** 0  
- **Frequency:** House has the highest frequency in `house_df`, covering approximately **50%** of the dataset.

#### Plot Dataset

- **Missing values:** 0  
- **Frequency:** Plot has the highest frequency in `plot_df`, covering approximately **98%** of the dataset.

## Is_ready_to_move

In [158]:
def bool_eda(df, col):

    print("="*60)
    print(f"{col.upper()} EDA")
    print("="*60)

    # Missing values
    print(f"\nMissing Values in {col} :", df[col].isnull().sum())

    # Value counts
    print(f"\nValue Counts of {col} :")
    print(df[col].value_counts())

    # Count plot
    plt.figure(figsize=(6,5))
    sns.countplot(x=df[col])

    plt.title(f"{col} Distribution")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.show()

    # Pie chart
    value_counts = df[col].value_counts()

    plt.figure(figsize=(6,6))
    plt.pie(
        value_counts.values,
        labels=value_counts.index,
        autopct='%1.1f%%'
    )

    plt.title(f"{col} Distribution (Percentage)")
    plt.show()

    return {
        "value_counts": value_counts
    }

In [162]:
# bool_eda(flat_df, col='is_ready_to_move')
# bool_eda(house_df, col='is_ready_to_move')
# bool_eda(plot_df, col='is_ready_to_move')

### Observations

- **Missing values:** 0 in apartment, house/villa, and plot datasets.

- **Apartment dataset:**  
  80% of properties are ready to move, while 20% are not.

- **House / Villa dataset:**  
  95% of properties are ready to move, while 5% are not.

- **Plot dataset:**  
  90% of properties are ready to move, while 10% are not.

## Area_Type

In [188]:
def area_type_combination_eda(df):

    cols = [
        "is_super_built_up_area",
        "is_built_up_area",
        "is_carpet_area"
    ]

    # Filter rows where at least one flag is True/1
    temp_df = df[df[cols].sum(axis=1) > 0]

    print("="*60)
    print("AREA TYPE COMBINATION EDA")
    print("="*60)

    print("\nFiltered Shape:", temp_df.shape)
    
    # missing values
    print("\nMissing Values in Area Type Flags:")
    print(house_df.shape[0]-temp_df.shape[0])

    # Value counts in percentage
    vc_df = (
        temp_df[cols]
        .value_counts(normalize=True)
        .mul(100)
        .reset_index(name="percentage")
    )

    print("\nCombination Percentage Distribution:")
    print(vc_df)

    # return temp_df, vc_df

In [191]:
# area_type_combination_eda(flat_df)
# area_type_combination_eda(house_df)
# area_type_combination_eda(plot_df)

### Observations

#### Area Type Features (`is_super_built_up_area`, `is_carpet_area`, `is_built_up_area`)

- **Missing values:**  
  - 45 missing values in the apartment dataset  
  - 210 missing values in the house/villa dataset  

- **Apartment dataset:**  
  `is_super_built_up_area` has the highest frequency, covering approximately **50%** of the dataset, followed by `is_carpet_area`.

- **House / Villa dataset:**  
  `is_built_up_area` has the highest frequency, covering approximately **47%** of the dataset, followed by `is_super_built_up_area`.

- **Plot dataset:**  
  These features are **irrelevant** for the plot dataset because plots represent land only, with no built‑up area.  
  → Therefore, drop these features from the plot dataset.